# Aggregate impact function inputs to polygons

For main analysis / figures:
- QPLAD `tassumpoly` to impact regions (Carleton uses 0.25-deg GMFD)
- QPLAD `tassumpoly` to ADM1s (BHM uses 0.5-deg UDel)
- QPLAD `tasmaxbinF` to US counties (Behrer uses 4km PRISM)

For resolution dependency verification, see `2b_aggregate_1deg_to_shps.ipynb`.

In [1]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_aux import _find_main_variable
dir_list = get_params()

In [2]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42699 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/42699/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/42699/status,Workers: 6
Total threads: 18,Total memory: 44.70 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39559,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/42699/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33545,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc/proxy/43113/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:46155,


In [3]:
# Get paths of BCD-ME files
bcd_fns = pd.DataFrame({'path':np.sort(glob.glob(dir_list['bcd_me']+'*.zarr'))})
bcd_fns['model'] = [re.split(r'\.',re.split(r'\_',fn)[-1])[0] for fn in bcd_fns.path]
bcd_fns['typ'] = [re.split(r'_',re.split(r'\/',fn)[-1])[2] for fn in bcd_fns.path.values]
modlist = bcd_fns['model'].unique()
bcd_fns = bcd_fns.set_index(['typ','model'])

In [4]:
params_proc = {'impact-regions':{'varnames':['tassumpoly'],'shp_fn':'impact-region'},
               'adm1':{'varnames':['tassumpoly'],'shp_fn':'country-fixed'},
               'adm2-us':{'varnames':['tasmaxbinF'],'shp_fn':'UScounties'}}


idxs = {'impact-regions':{'hierid':'locid'},
        'adm1':{'CNTRY_NAME':'locid'},
        'adm2-us':{'FIPS':'locid'}}

In [5]:
# Load population density
pop_fn = dir_list['aux']+'pop_data/gpw_v4_population_count_rev11_025deg_rg.nc'
pop = xr.open_dataset(pop_fn)

### Pre-processing of shapefiles
Primarily needed for BHM replication

In [7]:
if not os.path.exists(dir_list['aux']+'geo_data/country-fixed.shp'):
    
    # Load shapefile for insample regions
    gdf = gpd.read_file(dir_list['aux']+'geo_data/country.shp')
    
    rename_keys = {'Belarus':'Byelarus',
     'Brunei Darussalam':'Brunei',
     'Cabo Verde':'Cape Verde',
     'Channel Islands':['Jersey','Guernsey'],
     'Congo, Dem. Rep.':'Zaire',
     'Congo, Rep.':'Congo',
     "Cote d'Ivoire":'Ivory Coast',
     'Egypt, Arab Rep.':'Egypt',
     'Hong Kong SAR, China':'Hong Kong', #probably missing,
     'Iran, Islamic Rep.':'Iran',
     'Isle of Man':'Man, Isle of',
     'Korea, Rep.':'South Korea',
     'Kosovo':'Kosovo', # still part of Serbia in this sample
     'Kyrgyz Republic':'Kyrgyzstan',
     'Lao PDR':'Laos',
     'Macao SAR, China':'Macau',
     'Macedonia, FYR':'Macedonia',
     'Micronesia, Fed. Sts.':'Federated States of Micronesia',
     'Palau':'Pacific Islands (Palau)',
     'Russian Federation':'Russia',
     'Samoa':'Western Samoa',
     'Slovak Republic':'Slovakia',
     'Syrian Arab Republic':'Syria',
     'Tanzania':'Tanzania, United Republic of',
     'Timor-Leste':'Timor-Leste', # uh, still part of indonesia in this one... 
     'Venezuela, RB':'Venezuela',
     'West Bank and Gaza':['West Bank','Gaza Strip'],
     'Yemen, Rep.':'Yemen'}
    # Switch order
    rename_keys = {(tuple(v) if type(v) == list else v):k for k,v in rename_keys.items()}
    
    # Combine into single the ones that are multiple locs in the shapefile but not the repl dataset
    combine_locs = {k:v for k,v in rename_keys.items() if type(k) == tuple}
    for locs,joint_name in combine_locs.items():
        gdf_tmp = gdf.iloc[[c in locs for c in gdf.CNTRY_NAME.values],:]
        cols = [col for col in gdf_tmp.columns if col != 'geometry']
        
        gdf_joint = pd.concat([gdf_tmp.iloc[0].loc[[col for col in cols if gdf_tmp.dtypes[col] == object]],
                              gdf_tmp.loc[:,[col for col in cols if gdf_tmp.dtypes[col] in [float,int]]].sum()])
        gdf_joint = gpd.GeoDataFrame([gdf_joint],index=[gdf_tmp.index[0]],
                         geometry = [gdf_tmp.union_all()],
                                     crs = gdf_tmp.crs)
        gdf_joint.loc[:,'CNTRY_NAME'] = joint_name
        
        gdf = pd.concat([gdf.iloc[[c not in locs for c in gdf.CNTRY_NAME.values],:],
                   gdf_joint])
    gdf = gdf.sort_index()
    
    # Rename the rest of the locations as needed
    for idx in gdf.index:
        if gdf.loc[idx,'CNTRY_NAME'] in rename_keys:
            gdf.loc[idx,'CNTRY_NAME'] = rename_keys[gdf.loc[idx,'CNTRY_NAME']]
    
    # Save 
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore') # shp writing warning about POP_CNTRY stat
        gdf.to_file(dir_list['aux']+'geo_data/country-fixed.shp')

## Processing

In [ ]:
# Load sample QPLAD file
ds_grid = xr.open_zarr(bcd_fns.loc['qdm-qplad'].iloc[0]['path'],decode_timedelta=False)[['lat','lon']]

overwrite = True
# Create xagg weightmaps
for proc in ['impact-regions','adm1']:#params_proc:#['adm2-us']:#params_proc:
    #------------------------
    # Setup -----------------
    #------------------------
    # Generate filenames
    output_fns = {mod:{var:{'nowght':f'{dir_list['proc']}aggs/{var}_agg_{mod}_ALLRUNS_ALLGWLs_{proc}.zarr',
                            'popwght':f'{dir_list['proc']}aggs/{var}_agg_{mod}_ALLRUNS_ALLGWLs_{proc}_popwght.zarr'}
                   for var in params_proc[proc]['varnames']}
             for mod in modlist}

    # Check for completed existence of all files for a given 
    # shapefile (utility_print adds an empty .done flag after saving)
    if not overwrite:
        if np.all([np.all([np.all([os.path.exists(output_fns[k][v][t]+'/.done') 
                                   for t in output_fns[k][v]]) 
                           for v in output_fns[k]]) 
                   for k in output_fns.keys()]):
            print(f'All files generated for {proc}, skipped.')
            continue
    
    #------------------------
    # Generate weightmaps ---
    #------------------------
    # Load shapefile
    gdf = gpd.read_file(f'{dir_list['aux']}geo_data/{params_proc[proc]['shp_fn']}.shp')
    gdf['geometry'] = gdf.make_valid()

    # Keep relevant index columns
    gdf = gdf.loc[:,[*[k for k in idxs[proc]],'geometry']]

    # Rename index columns to standard names across analyses
    gdf = gdf.rename(idxs[proc],axis=1)

    # Create weightmap
    with warnings.catch_warnings():
        # Catches a weights bbox warning that doesn't affect this process
        warnings.filterwarnings('ignore')
        wms = {'nowght':xa.pixel_overlaps(ds_grid,gdf),
               'popwght':xa.pixel_overlaps(ds_grid,gdf,weights=pop.pop)}

    #------------------------
    # Aggregate by model ----
    #------------------------
    for mod in modlist:
        for var in params_proc[proc]['varnames']:
            # Check for completed existence of files for a given
            # model / variable combination (utility_print adds an
            # empty .done flag after saving)
            path_exists = {wght:os.path.exists(output_fns[mod][var][wght]+'/.done') for wght in output_fns[mod][var]}
            if not overwrite:
                if np.all([v for k,v in path_exists.items()]):
                    print(f'All files for {mod} {var} aggregated to {proc} already generated, skipped.')
                    continue
                proc_weights = [wght for wght,flag in path_exists.items() if not flag]
            else:
                proc_weights = [wght for wght in path_exists]
            
            # Load QDM-QPLAD downscaled data
            try:
                ds = xr.open_zarr(bcd_fns.loc[('qdm-qplad',mod),'path'],decode_timedelta=False)
            except:
                print("('qdm-qplad',"+mod+") not found in filenames, skipped.")
                continue
            if var in ds:
                has_data_tmp = ds[['has_data']]
                ds = ds[[var]]
            else:
                print(f'{var} not in {bcd_fns.loc[('qdm-qplad',mod),'path']}, skipped.')
                continue

        
            # Rechunk to allow geographic processing
            ds = ds.chunk({'lat':-1,'lon':-1,'proj_base':-1})

            # Workaround for a bug in xa exports with singleton dimensions
            single_idv_fix = False
            if ds.sizes['idv'] == 1:
                idv_tmp = ds[['model','experiment','run']]
                ds = ds.isel(idv=0)
                single_idv_fix = True

            # Aggregate
            with warnings.catch_warnings():
                # Stops inconsistent nan warning from xagg - which is just catching that
                # some gwls are all-nans for some idvs
                warnings.filterwarnings('ignore')
                
                with xa.set_options(impl='numba',silent=True):
                    aggs = {wght:xa.aggregate(ds,wms[wght])
                            for wght in proc_weights}
    
            # Move aggregated output to xarray dataset format
            aggs_out = {wght:agg.to_dataset()
                        for wght,agg in aggs.items()}
                    
            # Add experiment / run id back (secondary coordinates don't make it
            # through the xagg workflow) 
            for wght in aggs_out:
                if single_idv_fix:
                    aggs_out[wght] = xr.merge([aggs['nowght'].to_dataset(),
                                               idv_tmp,has_data_tmp])
                    aggs_out[wght][_find_main_variable(aggs_out[wght])] = aggs_out[wght][_find_main_variable(aggs_out[wght])].expand_dims({'idv':aggs_out[wght].idv})
                    aggs_out[wght] = aggs_out[wght].transpose(*['poly_idx','proj_base','gwl','idv',
                                                                *[k for k in aggs_out[wght].sizes if k not in ['poly_idx','proj_base','gwl','idv']]])

                else:
                    aggs_out[wght] = aggs_out[wght].reset_index('idv').assign_coords({var:ds[var] for var in ['experiment','run']})
    
            #------------------------
            # Save ----
            #------------------------
            for wght in aggs_out:
                # Metadata
                if wght == 'nowght':
                    weightdesc = ''
                elif wght == 'popwght':
                    weightdesc = ' using pop. weights'
                aggs_out[wght].attrs = {'SOURCE':'aggregate_to_shps.ipynb',
                                        'DESCRIPTION':f'BCD-ME {mod} {var} data, aggregated{weightdesc} to {re.sub(r'\-', ' ',proc)} polygons'}
    
                # Chunking for zarr saving
                chunk_dims = ['idv','proj_base']
                full_dims = [dim for dim in aggs_out[wght][var].sizes if dim not in chunk_dims]
                aggs_out[wght] = aggs_out[wght].chunk({**{dim:1 for dim in chunk_dims},
                          **{dim:-1 for dim in full_dims}}
                        )

                # Load coordinates, if not loaded (can otherwise cause write errors)
                for v in aggs_out[wght].coords:
                    aggs_out[wght][v] = aggs_out[wght][v].load()
    
                # Output as zarr
                utility_save(aggs_out[wght].drop_encoding(),
                             output_fns[mod][var][wght],
                             zarr_mode = 'w',
                             save_kwargs = {'zarr_format':3})
                
        

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: /glade/u/home/schwarzwald/projects/impact_uncertainty/aux_data/geo_data/impact-region.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
creating polygons for each pixel...
calculating overlaps between pixels and output polygons...
success!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg_CESM2-WACCM_ALLRUNS_ALLGWLs_impact-regions.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:485: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=11, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg_CESM2-WACCM_ALLRUNS_ALLGWLs_impact-regions.zarr saved!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg_CESM2-WACCM_ALLRUNS_ALLGWLs_impact-regions_popwght.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:485: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=11, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type

...

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg_CNRM-CM6-1_ALLRUNS_ALLGWLs_adm1_popwght.zarr saved!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg_CNRM-ESM2-1_ALLRUNS_ALLGWLs_adm1.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:485: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=11, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg_CNRM-ESM2-1_ALLRUNS_ALLGWLs_adm1.zarr saved!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg_CNRM-ESM2-1_ALLRUNS_ALLGWLs_adm1_popwght.zarr removed to allow overwrite!


/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:485: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=11, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg_CNRM-ESM2-1_ALLRUNS_ALLGWLs_adm1_popwght.zarr saved!


### Add "tasmeanpoly" files, for GDP per capita growth projections
These use mean annual T and squared mean annual T, instead of their local sums. Can calculate from just the `tassumpoly` files, since the first degree is just 365 * the mean annual T. 

In [18]:
proc = 'gdppc'
# Get filenames of aggregated climate predictors
fns = np.sort(glob.glob(f'{dir_list['proc']}aggs/tassumpoly_*_adm1*'))

In [20]:
overwrite = False
for fn in tqdm(fns):
    output_fn = re.sub('tassumpoly','tasmeanpoly',fn)
    if overwrite or not os.path.exists(output_fn):
        ds_tmp = xr.open_zarr(fn)
        ds_tmp = ds_tmp.sel(degree=[1,2])
        ds_tmp['tassumpoly'].loc[{'degree':1}] = ds_tmp['tassumpoly'].loc[{'degree':1}]/365
        ds_tmp['tassumpoly'].loc[{'degree':2}] = ds_tmp['tassumpoly'].loc[{'degree':1}]**2
        
        ds_tmp = ds_tmp.rename({'tassumpoly':'tasmeanpoly'})
        ds_tmp.attrs['DESCRIPTION2'] = 'mean annual T and squared mean annual T, in additional processing for GDP per capita growth impacts'
        
        utility_save(ds_tmp,output_fn,save_kwargs = {'mode':'w'})
    else:
        print(output_fn+' exists, skipped')

  0%|          | 0/114 [00:00<?, ?it/s]

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmeanpoly_agg-1deg_ACCESS-CM2_ALLRUNS_ALLGWLs_adm1.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmeanpoly_agg-1deg_ACCESS-CM2_ALLRUNS_ALLGWLs_adm1_popwght.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmeanpoly_agg-1deg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm1.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmeanpoly_agg-1deg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm1_popwght.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmeanpoly_agg-1deg_BCC-CSM2-MR_ALLRUNS_ALLGWLs_adm1.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmeanpoly_agg-1deg_BCC-CSM2-MR_ALLRUNS_ALLGWLs_adm1_popwght.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tasmeanpoly_agg-1deg_BCC-ESM1_ALLRUNS_ALLGWLs_adm1.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impa

### Add "days90plus" files, for payroll projections
Separate files with days > 90 degrees instead of just bin-days

In [21]:
proc = 'payroll'
# Get filenames of aggregated climate predictors
fns = np.sort(glob.glob(f'{dir_list['proc']}aggs/tasmaxbinF_*_adm2-us*'))


In [24]:
overwrite = False
for fn in tqdm(fns):
    output_fn = re.sub('tasmaxbinF','days90plus',fn)
    if overwrite or not os.path.exists(output_fn):

        ds_tmp = xr.open_zarr(fn)
        if 'binF' in ds_tmp.sizes:
            ds_tmp = ds_tmp.rename({'binF':'bin'})
    
        # Make sure the "all 0s" are nans (issue with nansum that means
        # that nanpixels are all 0s instead)
        ds_tmp['tasmaxbinF'] = ds_tmp['tasmaxbinF'].where(ds_tmp['tasmaxbinF'].sum('bin')>0)
        
        ds_tmp = ds_tmp.sel(bin=slice(90,None)).sum('bin',skipna=False).expand_dims({'binF':['days90plus']})
        ds_tmp['locid'] = ds_tmp['locid'].isel(binF=0)
        ds_tmp = ds_tmp.rename({'tasmaxbinF':'days90plus'})
        ds_tmp.attrs['DESCRIPTION2'] = 'sum over bins > 90F, in additional processing for payroll impacts'
        
        utility_save(ds_tmp,output_fn,save_kwargs = {'mode':'w'})
    else:
        print(output_fn+' exists, skipped')

  0%|          | 0/106 [00:00<?, ?it/s]

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/days90plus_agg-1deg_ACCESS-CM2_ALLRUNS_ALLGWLs_adm2-us.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/days90plus_agg-1deg_ACCESS-CM2_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/days90plus_agg-1deg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm2-us.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/days90plus_agg-1deg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/days90plus_agg-1deg_BCC-CSM2-MR_ALLRUNS_ALLGWLs_adm2-us.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/days90plus_agg-1deg_BCC-CSM2-MR_ALLRUNS_ALLGWLs_adm2-us_popwght.zarr exists, skipped
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/days90plus_agg-1deg_BCC-ESM1_ALLRUNS_ALLGWLs_adm2-us.zarr exists, skipped
/glade/derecho/scratch/sc